In [16]:
from datasets import Dataset
from pathlib import Path

def load_conll(filepath):
    with open(filepath, encoding="utf-8") as f:
        sentences = []
        labels = []
        current_tokens = []
        current_tags = []

        for line in f:
            line = line.strip()
            if not line:
                if current_tokens:
                    sentences.append(current_tokens)
                    labels.append(current_tags)
                    current_tokens, current_tags = [], []
                continue
            splits = line.split()
            if len(splits) >= 2:
                token = splits[0]
                tag = splits[-1]
                current_tokens.append(token)
                current_tags.append(tag)

        if current_tokens:
            sentences.append(current_tokens)
            labels.append(current_tags)

    return Dataset.from_dict({"tokens": sentences, "ner_tags": labels})

# Load your actual dataset
dataset = load_conll("../data/processed/ethiopic_news_ner.conll")
dataset = dataset.train_test_split(test_size=0.1)

dataset


DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 89
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 10
    })
})

In [17]:
from transformers import AutoTokenizer

model_checkpoint = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Confirm it supports fast tokenization (needed for alignment)
assert tokenizer.is_fast

In [18]:
# Flatten all tags across dataset to get label list
unique_tags = set(tag for seq in dataset["train"]["ner_tags"] for tag in seq)
label_list = sorted(list(unique_tags))
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

print(label2id)

{'B-LOC': 0, 'B-PRICE': 1, 'B-PRODUCT': 2, 'I-LOC': 3, 'I-PRICE': 4, 'I-PRODUCT': 5, 'O': 6}


In [19]:
def tokenize_and_align_labels(example):
    tokenized_inputs = tokenizer(example["tokens"],
                                 is_split_into_words=True,
                                 truncation=True,
                                 padding="max_length",
                                 max_length=128)

    word_ids = tokenized_inputs.word_ids()
    previous_word_idx = None
    label_ids = []

    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-100)
        elif word_idx != previous_word_idx:
            label_ids.append(label2id[example["ner_tags"][word_idx]])
        else:
            # For subword tokens: use I-... if previous was B-...
            prev_tag = example["ner_tags"][word_idx]
            if prev_tag.startswith("B-"):
                label_ids.append(label2id[prev_tag.replace("B-", "I-")])
            else:
                label_ids.append(label2id[prev_tag])
        previous_word_idx = word_idx

    tokenized_inputs["labels"] = label_ids
    return tokenized_inputs

In [20]:
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=False)
tokenized_dataset["train"][0]

Map: 100%|██████████| 10/10 [00:00<00:00, 2200.23 examples/s]


{'tokens': ['የከላይ',
  'ጫን',
  'ስደረግ',
  'ሳሙና',
  'እየሳብ',
  'ስፖንጅ',
  'ላይ',
  'የሚያደርግ',
  'የራሱ',
  'ስፕንጅ',
  'አብሮት',
  'ያለዉ',
  'ዋጋ',
  '550'],
 'ner_tags': ['B-PRODUCT',
  'I-PRODUCT',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'O',
  'B-PRICE',
  'I-PRICE'],
 'input_ids': [0,
  623,
  5679,
  68249,
  6,
  15741,
  548,
  17930,
  63488,
  32665,
  12273,
  946,
  13298,
  124590,
  17930,
  55714,
  548,
  12261,
  2269,
  13262,
  61860,
  623,
  109616,
  17930,
  32014,
  548,
  12261,
  181014,
  1437,
  16863,
  12131,
  80667,
  45334,
  2,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1

In [21]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    "xlm-roberta-base",
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id
)

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [22]:
import evaluate
from sklearn.metrics import classification_report

seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = predictions.argmax(axis=-1)

    true_labels = [[id2label[label] for label in sent if label != -100]
                   for sent in labels]
    true_preds = [[id2label[pred] for pred, label in zip(sent_preds, sent_labels) if label != -100]
                  for sent_preds, sent_labels in zip(predictions, labels)]

    results = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

In [23]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./ner_results",
    do_train=True,
    do_eval=True,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
)

In [24]:
# Define a simple accuracy metric
import evaluate
import torch
metric_accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = torch.argmax(torch.tensor(logits), dim=-1)
    return metric_accuracy.compute(predictions=predictions, references=labels)

In [25]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics
)


In [26]:
trainer.train()

/Users/mikiyasegaye/MK_Lab/10 Academy/Amharic-E-commerce-Data-Extractor/venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


TrainOutput(global_step=60, training_loss=0.6558270772298177, metrics={'train_runtime': 22.228, 'train_samples_per_second': 20.02, 'train_steps_per_second': 2.699, 'total_flos': 29070578254080.0, 'train_loss': 0.6558270772298177, 'epoch': 5.0})

In [27]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Save the model and tokenizer
model.save_pretrained("amharic-ner-model")
tokenizer.save_pretrained("amharic-ner-model")

('amharic-ner-model/tokenizer_config.json',
 'amharic-ner-model/special_tokens_map.json',
 'amharic-ner-model/tokenizer.json')

In [30]:
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

# Load fine-tuned model
model_path = "amharic-ner-model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)

# Create NER pipeline
ner = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

# Example message
amharic_text = "አዲስ ልብስ በቦሌ ላይ በ 250 ብር ሽያጭ ይካሄዳል"

# Run prediction
entities = ner(amharic_text)
for entity in entities:
    print(entity)


Device set to use mps:0
